# Phase 1: Source-Specific Data Cleaning & Standardization

**Project**: Context-Aware Generative Imputation of Air Pollution Data  
**Research Direction**: SLM-Conditioned Diffusion for Spatio-Temporal Missing Data Imputation  
**Author**: Antigravity Autonomous Research Assistant  

### Purpose & Scientific Boundaries
This notebook executes **Phase 1: Source-Specific Data Cleaning and Standardization** across the three independent raw data domains:
1. **Air Quality**: Hong Kong EPD hourly monitoring network (16 stations × 26,304 hours, 5 criteria pollutants)
2. **Meteorology**: Continuous surface atmospheric reanalysis at the 16 station coordinates (`rainfall` preserved)
3. **Traffic**: Transport Department 1st Generation Traffic Speed Map (`TRAFFIC_SPEED` & `ROAD_SATURATION_LEVEL`)

> **STRICT SCIENTIFIC PROTOCOL**:
> - `data/raw/` is **read-only** and cryptographically audited (SHA-256 pre and post).
> - Natural missingness in criteria pollutants is preserved strictly as `NaN` (no mean imputation, interpolation, or filling).
> - `rainfall` is maintained as our verified meteorological feature; it is **never** renamed to visibility.
> - The three sources remain **independent**; zero multimodal spatial/temporal merging, tensor construction, windowing, or modeling is performed in this phase.

## 1. Environment & Dependencies

In [1]:
import sys
import platform
from pathlib import Path
import json
import hashlib
import xml.etree.ElementTree as ET

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Print environment specifications
print(f'Python Version   : {platform.python_version()} ({platform.architecture()[0]})')
print(f'Pandas Version   : {pd.__version__}')
print(f'NumPy Version    : {np.__version__}')
print(f'Executable Path  : {sys.executable}')

Python Version   : 3.12.3 (64bit)
Pandas Version   : 3.0.5
NumPy Version    : 2.5.3
Executable Path  : /home/mocha/Desktop/ctdi-model-project/.venv/bin/python


## 2. Configuration & Paths
All paths are defined relative to the project root.

In [2]:
# Detect project root dynamically without hardcoded absolute paths
cwd = Path('.').resolve()
if cwd.name == 'notebooks':
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'

AIR_RAW_CSV = RAW_DIR / 'air_quality' / 'epd_air_quality_2019_2021_hourly.csv'
MET_RAW_CSV = RAW_DIR / 'meteorology' / 'hourly_meteorology_16stations_2019_2021.csv'
TRAFFIC_SAMPLES_DIR = RAW_DIR / 'traffic' / 'samples'

AIR_INTERIM_DIR = INTERIM_DIR / 'air_quality'
MET_INTERIM_DIR = INTERIM_DIR / 'meteorology'
TRAFFIC_INTERIM_DIR = INTERIM_DIR / 'traffic'
META_INTERIM_DIR = INTERIM_DIR / 'metadata'

# Verify existence of raw inputs
assert AIR_RAW_CSV.exists(), f'Missing {AIR_RAW_CSV}'
assert MET_RAW_CSV.exists(), f'Missing {MET_RAW_CSV}'
assert TRAFFIC_SAMPLES_DIR.exists(), f'Missing {TRAFFIC_SAMPLES_DIR}'

print('Configuration verified successfully.')
print(f'Project Root: {PROJECT_ROOT}')

Configuration verified successfully.
Project Root: /home/mocha/Desktop/ctdi-model-project


## 3. Air-Quality Loading
Ingesting the verified Hong Kong EPD hourly observation dataset.

In [3]:
print(f'Loading raw air quality data from: {AIR_RAW_CSV.relative_to(PROJECT_ROOT)}')
df_air_raw = pd.read_csv(AIR_RAW_CSV)
print(f'Initial Shape: {df_air_raw.shape[0]:,} rows × {df_air_raw.shape[1]} columns')
print(f'Raw Columns  : {df_air_raw.columns.tolist()}')
df_air_raw.head(3)

Loading raw air quality data from: data/raw/air_quality/epd_air_quality_2019_2021_hourly.csv


Initial Shape: 420,864 rows × 10 columns
Raw Columns  : ['station_id', 'station_name', 'timestamp', 'pm25', 'pm10', 'no2', 'o3', 'so2', 'nox', 'co']


,station_id,station_name,timestamp,pm25,pm10,no2,o3,so2,nox,co
0,71,CAUSEWAY BAY,2019-01-01 00:00:00,31.0,42.0,67.0,2.0,9.0,299.0,93.0
1,71,CAUSEWAY BAY,2019-01-01 01:00:00,31.0,41.0,60.0,3.0,5.0,184.0,90.0
2,71,CAUSEWAY BAY,2019-01-01 02:00:00,27.0,36.0,40.0,7.0,3.0,70.0,80.0


## 4. Air-Quality Validation
Validating Cartesian grid completeness, unique stations, timestamps, and natural missingness.

In [4]:
EXPECTED_STATIONS = 16
EXPECTED_TIMESTAMPS = 26304
EXPECTED_TOTAL_ROWS = EXPECTED_STATIONS * EXPECTED_TIMESTAMPS  # 420,864

# Grid assertions
n_stations = df_air_raw['station_id'].nunique()
n_timestamps = df_air_raw['timestamp'].nunique()
n_rows = len(df_air_raw)

assert n_stations == EXPECTED_STATIONS, f'Expected {EXPECTED_STATIONS} stations, found {n_stations}'
assert n_timestamps == EXPECTED_TIMESTAMPS, f'Expected {EXPECTED_TIMESTAMPS} timestamps, found {n_timestamps}'
assert n_rows == EXPECTED_TOTAL_ROWS, f'Expected {EXPECTED_TOTAL_ROWS} rows, found {n_rows}'

# Duplicate record assertion
n_dups = df_air_raw.duplicated(subset=['station_id', 'timestamp']).sum()
assert n_dups == 0, f'Found {n_dups} duplicate records'

# Missing values breakdown across the 5 criteria pollutants
POLLUTANTS = ['pm25', 'pm10', 'no2', 'so2', 'o3']
missing_counts = {p: int(df_air_raw[p].isna().sum()) for p in POLLUTANTS}
total_missing = sum(missing_counts.values())

print('=== AIR QUALITY VALIDATION SUMMARY ===')
print(f'Stations             : {n_stations} (13 general + 3 roadside)')
print(f'Timestamps           : {n_timestamps:,} continuous hours (2019-01-01 to 2021-12-31)')
print(f'Total Grid Records   : {n_rows:,}')
print(f'Duplicates Detected  : {n_dups}')
print('Criteria Pollutant Missingness:')
for p in POLLUTANTS:
    pct = missing_counts[p] / n_rows * 100
    print(f'  - {p:6s}: {missing_counts[p]:6,d} missing ({pct:.2f}%)')
print(f'Total Missing Values : {total_missing:,} (Matches CTDI reference: 55,876)')

# Check non-negativity
for p in POLLUTANTS:
    neg = (df_air_raw[p] < 0.0).sum()
    assert neg == 0, f'Found {neg} negative values in {p}'
print('All concentration values >= 0.0 (Zero physical boundary violations).')

=== AIR QUALITY VALIDATION SUMMARY ===
Stations             : 16 (13 general + 3 roadside)
Timestamps           : 26,304 continuous hours (2019-01-01 to 2021-12-31)
Total Grid Records   : 420,864
Duplicates Detected  : 0
Criteria Pollutant Missingness:
  - pm25  : 10,657 missing (2.53%)
  - pm10  : 11,395 missing (2.71%)
  - no2   : 11,651 missing (2.77%)
  - so2   : 11,056 missing (2.63%)
  - o3    : 11,117 missing (2.64%)
Total Missing Values : 55,876 (Matches CTDI reference: 55,876)
All concentration values >= 0.0 (Zero physical boundary violations).


## 5. Air-Quality Cleaning & Standardization
Standardizing column names, ISO datetime parsing, chronological sorting, and station identifier normalization.

In [5]:
# Clean and standardize air quality
df_air_clean = df_air_raw.copy()
df_air_clean.columns = [c.strip().lower().replace('/', '_').replace(' ', '_') for c in df_air_clean.columns]

# Standardize data types
df_air_clean['station_id'] = df_air_clean['station_id'].astype(int)
df_air_clean['station_name'] = df_air_clean['station_name'].astype(str).str.strip().str.upper()
df_air_clean['timestamp'] = pd.to_datetime(df_air_clean['timestamp'])

# Enforce chronological and station sorting
df_air_clean = df_air_clean.sort_values(by=['timestamp', 'station_id']).reset_index(drop=True)

# Retain canonical columns and format timestamp
CANONICAL_AIR_COLS = ['station_id', 'station_name', 'timestamp', 'pm25', 'pm10', 'no2', 'so2', 'o3']
df_air_clean = df_air_clean[CANONICAL_AIR_COLS].copy()
df_air_clean['timestamp'] = df_air_clean['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

print('Clean Air Quality Dataset Formatted:')
display(df_air_clean.head(4))
print(f'Clean shape: {df_air_clean.shape}')

Clean Air Quality Dataset Formatted:


,station_id,station_name,timestamp,pm25,pm10,no2,so2,o3
0,66,SHAM SHUI PO,2019-01-01 00:00:00,20.0,27.0,28.0,3.0,23.0
1,69,TAI PO,2019-01-01 00:00:00,19.0,30.0,28.0,2.0,16.0
2,70,YUEN LONG,2019-01-01 00:00:00,29.0,37.0,NaN,NaN,NaN
3,71,CAUSEWAY BAY,2019-01-01 00:00:00,31.0,42.0,67.0,9.0,2.0


Clean shape: (420864, 8)


## 6. Air-Quality Export
Exporting clean interim dataset to Parquet, CSV, and generating validation report.

In [6]:
from src.preprocessing.clean_air_quality import clean_air_quality

# Re-run via authoritative module to guarantee complete serialization and report generation
df_air_exported, report_air = clean_air_quality(AIR_RAW_CSV, AIR_INTERIM_DIR)
print(f'Exported to: {AIR_INTERIM_DIR.relative_to(PROJECT_ROOT)}')
print(f'Parquet size: {(AIR_INTERIM_DIR / "clean_air_quality.parquet").stat().st_size:,} bytes')
print(f'CSV size    : {(AIR_INTERIM_DIR / "clean_air_quality.csv").stat().st_size:,} bytes')
print(f'Report file : {(AIR_INTERIM_DIR / "air_quality_validation.json").name}')

[Air Quality] Loading raw data from /home/mocha/Desktop/ctdi-model-project/data/raw/air_quality/epd_air_quality_2019_2021_hourly.csv...


[Air Quality] Exporting clean dataset to /home/mocha/Desktop/ctdi-model-project/data/interim/air_quality/clean_air_quality.parquet...
[Air Quality] Exporting clean dataset to /home/mocha/Desktop/ctdi-model-project/data/interim/air_quality/clean_air_quality.csv...


[Air Quality] Validation report saved to /home/mocha/Desktop/ctdi-model-project/data/interim/air_quality/air_quality_validation.json.
Exported to: data/interim/air_quality
Parquet size: 2,332,790 bytes
CSV size    : 23,753,080 bytes
Report file : air_quality_validation.json


## 7. Meteorology Loading
Ingesting the hourly surface meteorological dataset for the 16 station coordinates.

In [7]:
print(f'Loading raw meteorology from: {MET_RAW_CSV.relative_to(PROJECT_ROOT)}')
df_met_raw = pd.read_csv(MET_RAW_CSV)
print(f'Initial Shape: {df_met_raw.shape[0]:,} rows × {df_met_raw.shape[1]} columns')
print(f'Raw Columns  : {df_met_raw.columns.tolist()}')
df_met_raw.head(3)

Loading raw meteorology from: data/raw/meteorology/hourly_meteorology_16stations_2019_2021.csv
Initial Shape: 420,864 rows × 9 columns
Raw Columns  : ['station_id', 'station_name', 'timestamp', 'temperature', 'relative_humidity', 'wind_speed', 'wind_direction', 'pressure', 'rainfall']


,station_id,station_name,timestamp,temperature,relative_humidity,wind_speed,wind_direction,pressure,rainfall
0,71,CAUSEWAY BAY,2019-01-01 00:00:00,8.6,72,5.33,6,1025.0,0.0
1,71,CAUSEWAY BAY,2019-01-01 01:00:00,8.3,73,5.47,9,1024.8,0.0
2,71,CAUSEWAY BAY,2019-01-01 02:00:00,7.9,74,5.69,10,1024.2,0.0


## 8. Meteorology Validation
Validating continuous coverage, absence of missing values, physical ranges, and the verified presence of `rainfall` (NEVER visibility).

In [8]:
assert 'visibility' not in df_met_raw.columns, 'ERROR: Found illegal visibility column!'
assert 'rainfall' in df_met_raw.columns, 'ERROR: Missing canonical rainfall column!'

n_met_stations = df_met_raw['station_id'].nunique()
n_met_timestamps = df_met_raw['timestamp'].nunique()
assert n_met_stations == EXPECTED_STATIONS
assert n_met_timestamps == EXPECTED_TIMESTAMPS
assert len(df_met_raw) == EXPECTED_TOTAL_ROWS

print('=== METEOROLOGY RANGE VALIDATION ===')
MET_VARS = ['temperature', 'relative_humidity', 'wind_speed', 'wind_direction', 'pressure', 'rainfall']
for var in MET_VARS:
    s = df_met_raw[var]
    assert s.isna().sum() == 0, f'Missing values in {var}'
    print(f'{var:18s} | min: {s.min():7.2f} | max: {s.max():7.2f} | mean: {s.mean():7.2f} | NaNs: {s.isna().sum()}')

assert (df_met_raw['rainfall'] >= 0.0).all(), 'Found negative rainfall!'
assert df_met_raw['relative_humidity'].between(0.0, 100.0).all(), 'RH out of bounds!'
assert df_met_raw['wind_direction'].between(0.0, 360.0).all(), 'WD out of bounds!'
print('All meteorological variables pass strict physical plausibility bounds.')

=== METEOROLOGY RANGE VALIDATION ===
temperature        | min:    2.90 | max:   35.60 | mean:   23.14 | NaNs: 0
relative_humidity  | min:   13.00 | max:  100.00 | mean:   81.54 | NaNs: 0
wind_speed         | min:    0.00 | max:   17.35 | mean:    3.54 | NaNs: 0
wind_direction     | min:    0.00 | max:  360.00 | mean:  120.34 | NaNs: 0
pressure           | min:  986.50 | max: 1029.90 | mean: 1010.51 | NaNs: 0
rainfall           | min:    0.00 | max:   61.80 | mean:    0.24 | NaNs: 0
All meteorological variables pass strict physical plausibility bounds.


## 9. Meteorology Cleaning & Standardization
Column standardization to canonical names, ISO datetime parsing, chronological sorting, and unit preservation.

In [9]:
CANONICAL_MET_COLS = ['station_id', 'station_name', 'timestamp', 'temperature', 'relative_humidity', 'pressure', 'rainfall', 'wind_direction', 'wind_speed']
df_met_clean = df_met_raw.copy()
df_met_clean['station_id'] = df_met_clean['station_id'].astype(int)
df_met_clean['station_name'] = df_met_clean['station_name'].astype(str).str.strip().str.upper()
df_met_clean['timestamp'] = pd.to_datetime(df_met_clean['timestamp'])
df_met_clean = df_met_clean.sort_values(by=['timestamp', 'station_id']).reset_index(drop=True)
df_met_clean = df_met_clean[CANONICAL_MET_COLS].copy()
df_met_clean['timestamp'] = df_met_clean['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

print('Clean Meteorology Dataset Formatted:')
display(df_met_clean.head(4))
print(f'Clean shape: {df_met_clean.shape}')

Clean Meteorology Dataset Formatted:


,station_id,station_name,timestamp,temperature,relative_humidity,pressure,rainfall,wind_direction,wind_speed
0,66,SHAM SHUI PO,2019-01-01 00:00:00,8.1,71,1026.4,0.0,6,5.33
1,69,TAI PO,2019-01-01 00:00:00,7.5,75,1027.9,0.0,20,3.29
2,70,YUEN LONG,2019-01-01 00:00:00,7.8,70,1027.7,0.0,23,4.79
3,71,CAUSEWAY BAY,2019-01-01 00:00:00,8.6,72,1025.0,0.0,6,5.33


Clean shape: (420864, 9)


## 10. Meteorology Export
Exporting clean interim meteorology dataset to Parquet, CSV, and generating validation report.

In [10]:
from src.preprocessing.clean_meteorology import clean_meteorology

df_met_exported, report_met = clean_meteorology(MET_RAW_CSV, MET_INTERIM_DIR)
print(f'Exported to: {MET_INTERIM_DIR.relative_to(PROJECT_ROOT)}')
print(f'Parquet size: {(MET_INTERIM_DIR / "clean_meteorology.parquet").stat().st_size:,} bytes')
print(f'CSV size    : {(MET_INTERIM_DIR / "clean_meteorology.csv").stat().st_size:,} bytes')
print(f'Report file : {(MET_INTERIM_DIR / "meteorology_validation.json").name}')

[Meteorology] Loading raw data from /home/mocha/Desktop/ctdi-model-project/data/raw/meteorology/hourly_meteorology_16stations_2019_2021.csv...


[Meteorology] Exporting clean dataset to /home/mocha/Desktop/ctdi-model-project/data/interim/meteorology/clean_meteorology.parquet...
[Meteorology] Exporting clean dataset to /home/mocha/Desktop/ctdi-model-project/data/interim/meteorology/clean_meteorology.csv...


[Meteorology] Validation report saved to /home/mocha/Desktop/ctdi-model-project/data/interim/meteorology/meteorology_validation.json.
Exported to: data/interim/meteorology
Parquet size: 2,652,146 bytes
CSV size    : 25,460,811 bytes
Report file : meteorology_validation.json


## 11. Traffic Loading
Safely parsing raw 1st Generation Traffic Speed Map XML snapshots using ElementTree.

In [11]:
from src.preprocessing.clean_traffic import parse_speedmap_xml

xml_files = sorted(list(TRAFFIC_SAMPLES_DIR.glob('historical_td_speedmap_*.xml')))
print(f'Found {len(xml_files)} representative Speedmap XML snapshots in: {TRAFFIC_SAMPLES_DIR.relative_to(PROJECT_ROOT)}')

raw_traffic_records = []
for xf in xml_files:
    recs = parse_speedmap_xml(xf)
    print(f'  - {xf.name}: {len(recs):4d} records extracted ({xf.stat().st_size:,} bytes)')
    raw_traffic_records.extend(recs)

df_traffic_raw = pd.DataFrame(raw_traffic_records)
print(f'Total Traffic Records Loaded: {len(df_traffic_raw):,}')
df_traffic_raw.head(3)

Found 4 representative Speedmap XML snapshots in: data/raw/traffic/samples
  - historical_td_speedmap_20190101_0000.xml:  607 records extracted (198,618 bytes)
  - historical_td_speedmap_20200101_0000.xml:  590 records extracted (193,129 bytes)
  - historical_td_speedmap_20210101_0000.xml:  608 records extracted (198,956 bytes)
  - historical_td_speedmap_20211231_2357.xml:  608 records extracted (198,937 bytes)
Total Traffic Records Loaded: 2,413


,snapshot_file,link_id,traffic_speed_raw,road_saturation_level_raw,capture_date_raw,region,road_type
0,historical_td_speedmap_20190101_0000.xml,3006-30069,46,TRAFFIC GOOD,2018-12-31T23:56:35,K,URBAN ROAD
1,historical_td_speedmap_20190101_0000.xml,30069-888301,49,TRAFFIC GOOD,2018-12-31T23:56:35,K,URBAN ROAD
2,historical_td_speedmap_20190101_0000.xml,3010-888301,43,TRAFFIC GOOD,2018-12-31T23:56:35,K,URBAN ROAD


## 12. Traffic Validation
Auditing road link counts, speed distributions, timestamps, and saturation level categories.

In [12]:
# Parse speeds
speeds = pd.to_numeric(df_traffic_raw['traffic_speed_raw'], errors='coerce')
assert speeds.isna().sum() == 0, 'Found NaN speeds'
assert (speeds >= 0.0).all(), 'Found negative speeds'
assert (speeds <= 200.0).all(), 'Found speeds > 200 km/h'

# Road link counts
unique_links = df_traffic_raw['link_id'].unique()
print('=== TRAFFIC VALIDATION SUMMARY ===')
print(f'Total Snapshots Audited : {len(xml_files)}')
print(f'Unique Road Links Union : {len(unique_links)} (Baseline: 607 links in 2019 snapshot)')
print(f'Speed Range (km/h)      : [{speeds.min():.1f}, {speeds.max():.1f}] | Mean: {speeds.mean():.1f} km/h')
print('Saturation Category Counts:')
display(df_traffic_raw['road_saturation_level_raw'].value_counts())

=== TRAFFIC VALIDATION SUMMARY ===
Total Snapshots Audited : 4
Unique Road Links Union : 632 (Baseline: 607 links in 2019 snapshot)
Speed Range (km/h)      : [3.0, 109.0] | Mean: 61.8 km/h
Saturation Category Counts:


road_saturation_level_raw
TRAFFIC GOOD       2130
TRAFFIC AVERAGE     244
TRAFFIC BAD          39
Name: count, dtype: int64

## 13. Traffic Cleaning & Ordinal Congestion Specification
Preserving raw saturation categories while defining and recording an explicit continuous ordinal mapping:
- `TRAFFIC GOOD`: 0.0
- `TRAFFIC AVERAGE`: 0.5
- `TRAFFIC BAD`: 1.0

> **Note**: Zero spatial IDW or station aggregation is performed in this phase.

In [13]:
ORDINAL_MAP = {'TRAFFIC GOOD': 0.0, 'TRAFFIC AVERAGE': 0.5, 'TRAFFIC BAD': 1.0}

df_traffic_clean = df_traffic_raw.copy()
df_traffic_clean['traffic_speed'] = pd.to_numeric(df_traffic_clean['traffic_speed_raw'])
df_traffic_clean['traffic_congestion'] = df_traffic_clean['road_saturation_level_raw'].str.strip().str.upper()
df_traffic_clean['traffic_congestion_ordinal'] = df_traffic_clean['traffic_congestion'].map(ORDINAL_MAP)
df_traffic_clean['capture_timestamp'] = pd.to_datetime(df_traffic_clean['capture_date_raw']).dt.strftime('%Y-%m-%d %H:%M:%S')

TRAFFIC_CLEAN_COLS = ['snapshot_file', 'link_id', 'traffic_speed', 'traffic_congestion', 'traffic_congestion_ordinal', 'capture_timestamp', 'region', 'road_type']
df_traffic_clean = df_traffic_clean[TRAFFIC_CLEAN_COLS].copy()

print('Clean Traffic Dataset Formatted:')
display(df_traffic_clean.head(4))
print(f'Clean shape: {df_traffic_clean.shape}')

Clean Traffic Dataset Formatted:


,snapshot_file,link_id,traffic_speed,traffic_congestion,traffic_congestion_ordinal,capture_timestamp,region,road_type
0,historical_td_speedmap_20190101_0000.xml,3006-30069,46,TRAFFIC GOOD,0.0,2018-12-31 23:56:35,K,URBAN ROAD
1,historical_td_speedmap_20190101_0000.xml,30069-888301,49,TRAFFIC GOOD,0.0,2018-12-31 23:56:35,K,URBAN ROAD
2,historical_td_speedmap_20190101_0000.xml,3010-888301,43,TRAFFIC GOOD,0.0,2018-12-31 23:56:35,K,URBAN ROAD
3,historical_td_speedmap_20190101_0000.xml,3363-3369,33,TRAFFIC GOOD,0.0,2018-12-31 23:56:35,K,URBAN ROAD


Clean shape: (2413, 8)


## 14. Traffic Export
Exporting clean traffic snapshots to Parquet, CSV, and generating validation audit report.

In [14]:
from src.preprocessing.clean_traffic import clean_traffic

df_traffic_exported, report_traffic = clean_traffic(TRAFFIC_SAMPLES_DIR, TRAFFIC_INTERIM_DIR)
print(f'Exported to: {TRAFFIC_INTERIM_DIR.relative_to(PROJECT_ROOT)}')
print(f'Parquet size: {(TRAFFIC_INTERIM_DIR / "clean_traffic_speedmap_snapshots.parquet").stat().st_size:,} bytes')
print(f'CSV size    : {(TRAFFIC_INTERIM_DIR / "clean_traffic_speedmap_snapshots.csv").stat().st_size:,} bytes')
print(f'Report file : {(TRAFFIC_INTERIM_DIR / "traffic_source_cleaning_report.json").name}')

[Traffic] Processing 4 raw XML snapshots from /home/mocha/Desktop/ctdi-model-project/data/raw/traffic/samples...
[Traffic] Exporting clean dataset to /home/mocha/Desktop/ctdi-model-project/data/interim/traffic/clean_traffic_speedmap_snapshots.parquet...
[Traffic] Exporting clean dataset to /home/mocha/Desktop/ctdi-model-project/data/interim/traffic/clean_traffic_speedmap_snapshots.csv...
[Traffic] Validation report saved to /home/mocha/Desktop/ctdi-model-project/data/interim/traffic/traffic_source_cleaning_report.json.
Exported to: data/interim/traffic
Parquet size: 17,423 bytes
CSV size    : 262,635 bytes
Report file : traffic_source_cleaning_report.json


## 15. Cross-Source Summary
Independent verification table across the three sources before any multimodal fusion.

In [15]:
from src.preprocessing.build_provenance import build_provenance_metadata

# Generate master provenance record
provenance_doc = build_provenance_metadata(PROJECT_ROOT, META_INTERIM_DIR / 'provenance_metadata.json')

summary_table = pd.DataFrame([
    {
        'Source Domain': 'Air Quality',
        'Target Entities': '16 monitoring stations',
        'Temporal Scope': '2019-01-01 to 2021-12-31 (26,304 hrs)',
        'Clean Rows': len(df_air_exported),
        'Key Variables': 'pm25, pm10, no2, so2, o3',
        'Missing Values': f'{report_air["total_criteria_pollutant_missing"]:,} NaNs (preserved)',
        'Status': 'CLEAN_STANDARDIZED'
    },
    {
        'Source Domain': 'Meteorology',
        'Target Entities': '16 station coordinates',
        'Temporal Scope': '2019-01-01 to 2021-12-31 (26,304 hrs)',
        'Clean Rows': len(df_met_exported),
        'Key Variables': 'temp, RH, press, rainfall, WD, WS',
        'Missing Values': '0 NaNs (continuous surface grid)',
        'Status': 'CLEAN_STANDARDIZED'
    },
    {
        'Source Domain': 'Traffic',
        'Target Entities': '607–632 road links',
        'Temporal Scope': 'Representative 2019-2021 snapshots',
        'Clean Rows': len(df_traffic_exported),
        'Key Variables': 'traffic_speed, traffic_congestion',
        'Missing Values': '0 NaNs',
        'Status': 'CLEAN_STANDARDIZED'
    }
])

print('=== INDEPENDENT SOURCE SUMMARY (ZERO PREMATURE MERGING) ===')
display(summary_table)
print(f'Provenance saved to: {(META_INTERIM_DIR / "provenance_metadata.json").relative_to(PROJECT_ROOT)}')

=== INDEPENDENT SOURCE SUMMARY (ZERO PREMATURE MERGING) ===


,Source Domain,Target Entities,Temporal Scope,Clean Rows,Key Variables,Missing Values,Status
0,Air Quality,16 monitoring stations,"2019-01-01 to 2021-12-31 (26,304 hrs)",420864,"pm25, pm10, no2, so2, o3","55,876 NaNs (preserved)",CLEAN_STANDARDIZED
1,Meteorology,16 station coordinates,"2019-01-01 to 2021-12-31 (26,304 hrs)",420864,"temp, RH, press, rainfall, WD, WS",0 NaNs (continuous surface grid),CLEAN_STANDARDIZED
2,Traffic,607–632 road links,Representative 2019-2021 snapshots,2413,"traffic_speed, traffic_congestion",0 NaNs,CLEAN_STANDARDIZED


Provenance saved to: data/interim/metadata/provenance_metadata.json


## 16. Raw-Data Integrity Check
Computing post-execution SHA-256 cryptographic hashes for all files in `data/raw/` and asserting zero modifications, zero deletions, and zero additions.

In [16]:
pre_manifest_file = META_INTERIM_DIR / 'raw_data_pre_manifest.json'
with open(pre_manifest_file, 'r', encoding='utf-8') as f:
    pre_manifest = json.load(f)

# Recompute current raw hashes
raw_files = sorted([p for p in RAW_DIR.glob('**/*') if p.is_file()])
current_manifest = {}
for p in raw_files:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    rel = str(p.relative_to(RAW_DIR))
    current_manifest[rel] = {'sha256': h.hexdigest(), 'size_bytes': p.stat().st_size}

# Calculate deltas
modified_files = []
for rel, info in pre_manifest.items():
    if rel not in current_manifest:
        pass # deleted
    elif current_manifest[rel]['sha256'] != info['sha256']:
        modified_files.append(rel)

deleted_files = set(pre_manifest.keys()) - set(current_manifest.keys())
added_files = set(current_manifest.keys()) - set(pre_manifest.keys())

print('=== RAW DATA CRYPTOGRAPHIC AUDIT RESULTS ===')
print(f'Total Raw Files Audited: {len(current_manifest)}')
print(f'Modified Raw Files     : {len(modified_files)}')
print(f'Deleted Raw Files      : {len(deleted_files)}')
print(f'Added Raw Files        : {len(added_files)}')

assert len(modified_files) == 0, f'Raw files were modified: {modified_files}'
assert len(deleted_files) == 0, f'Raw files were deleted: {deleted_files}'
assert len(added_files) == 0, f'Raw files were added: {added_files}'

print('>>> SUCCESS: 100% CRYPTOGRAPHIC RAW DATA IMMUTABILITY VERIFIED BIT-FOR-BIT! <<<')

=== RAW DATA CRYPTOGRAPHIC AUDIT RESULTS ===
Total Raw Files Audited: 683
Modified Raw Files     : 0
Deleted Raw Files      : 0
Added Raw Files        : 0
>>> SUCCESS: 100% CRYPTOGRAPHIC RAW DATA IMMUTABILITY VERIFIED BIT-FOR-BIT! <<<
